In [2]:
# Force the notebook to download the data from Nathaniel Miller's public GitHub repository at launch
!wget https://raw.githubusercontent.com/nathaniel-a-miller/arabic-poetry-dh/main/data/0198AbuNuwas.txt
!wget https://raw.githubusercontent.com/nathaniel-a-miller/arabic-poetry-dh/main/data/0211AbuAtahiya.txt

--2026-06-15 18:46:18--  https://raw.githubusercontent.com/nathaniel-a-miller/arabic-poetry-dh/main/data/0198AbuNuwas.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 618453 (604K) [text/plain]
Saving to: ‘0198AbuNuwas.txt’

0198AbuNuwas.txt    100%[===================>] 603.96K  --.-KB/s    in 0.05s   

2026-06-15 18:46:19 (12.1 MB/s) - ‘0198AbuNuwas.txt’ saved [618453/618453]

--2026-06-15 18:46:19--  https://raw.githubusercontent.com/nathaniel-a-miller/arabic-poetry-dh/main/data/0211AbuAtahiya.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting respon

In [ ]:
# 1. Install required rendering packages for Arabic right-to-left plotting
!pip install arabic-reshaper python-bidi matplotlib seaborn -q

# 2. Download raw text files directly from GitHub repository
import urllib.request
import os

base_url = "https://raw.githubusercontent.com/nathaniel-a-miller/arabic-poetry-dh/main/data/"
files = ["0198AbuNuwas.txt", "0211AbuAtahiya.txt"]

for f in files:
    try:
        urllib.request.urlretrieve(base_url + f, f)
        print(f"Successfully cached file locally: {f}")
    except Exception as e:
        print(f"Error downloading {f}. Check your repository visibility and URL structure. Error: {e}")

In [ ]:
from collections import Counter
import re

STOP_WORDS = {"من", "في", "على", "إلى", "و", "أن", "إن", "ما", "لا", "عن", "هي", "بـ", "إلا", "لـ", "ثم", "أو"}

def parse_and_count(file_path, filter_stops=True):
    text_lines = []
    in_header = True

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Could not find local file: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if "#META# Header#End#" in line:
                in_header = False
                continue
            if not in_header:
                text_lines.append(line)

    full_text = " ".join(text_lines)

    # Strip tashkil (diacritics) and tatweel (kashida extension character)
    sanitized = re.sub(r"[\u064B-\u0652\u0640]", "", full_text)

    # Isolate individual word tokens
    words = re.findall(r"\b\w+\b", sanitized)

    # Apply conditional stop word filtration mask
    if filter_stops:
        words = [w for w in words if w not in STOP_WORDS]

    return Counter(words)

In [ ]:
# Run calculation on the files downloaded in Cell 1
# You can toggle filter_stops to True or False to see the difference
nuwas_counts = parse_and_count("0198AbuNuwas.txt", filter_stops=True)
atah_counts = parse_and_count("0211AbuAtahiya.txt", filter_stops=True)

# Define how many top frequent terms to isolate dynamically
top_n = 5

print(f"--- DYNAMIC TOP {top_n} WORDS: ABU NUWAS ---")
for word, count in nuwas_counts.most_common(top_n):
    print(f"Token: {word} | Count: {count}")

print(f"\n--- DYNAMIC TOP {top_n} WORDS: ABU AL-ATAHIYAH ---")
for word, count in atah_counts.most_common(top_n):
    print(f"Token: {word} | Count: {count}")

In [ ]:
from bidi.algorithm import get_display
import arabic_reshaper
import matplotlib.pyplot as plt

def plot_top_words(counter_obj, title, color):
    # Pull out the actual top records dynamically computed from the text
    top_data = counter_obj.most_common(5)
    if not top_data:
        print("No data available to plot.")
        return

    words, counts = zip(*top_data)

    # Reshape and flip strings so Matplotlib handles Arabic layout correctly
    reshaped_labels = [get_display(arabic_reshaper.reshape(w)) for w in words]

    plt.figure(figsize=(8, 4))
    plt.bar(reshaped_labels, counts, color=color)
    plt.ylabel("Absolute Occurrences")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Render both plots dynamically
plot_top_words(nuwas_counts, "Empirical Top Vocabulary: Abu Nuwas", "maroon")
plot_top_words(atah_counts, "Empirical Top Vocabulary: Abu al-Atahiyah", "teal")